# 预训练循环(精简版)

> [ch08.ipynb](./ch08.ipynb) 的浓缩版。

## 8 个训练脚本共用骨架

```
init_distributed → config → init_model → dataset → AdamW+get_lr → DDP → epoch loop → checkpoint
```

## 关键参数

| 参数 | 预训练 | SFT | LoRA |
|---|---|---|---|
| lr | 5e-4 | 1e-5 | 1e-4 |
| from_weight | none | pretrain | full_sft |
| max_seq_len | 340 | 768 | 768 |
| 数据 | pretrain_t2t | sft_t2t | lora_medical |

## 余弦 LR

```python
lr = base_lr * (0.1 + 0.45 * (1 + cos(π * step / total)))
```

## 最小训练循环

```python
model = MiniMindForCausalLM(config).cuda()
optimizer = AdamW(model.parameters(), lr=5e-4)
for epoch in range(2):
    for batch in dataloader:
        with autocast(dtype=torch.bfloat16):
            loss = model(batch['input_ids'], labels=batch['labels'])
        loss.backward()
        clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        optimizer.zero_grad()
```